[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/fastapi-certified/notebooks/day-02-path-query-body.ipynb#scrollTo=10a2b3c4)

---
# Day 2 · Path, Query, and Body Parameters
**certified-journeys / fastapi-certified** · Day 2 · Core Parameter Handling

> **Goal for today:** Master the three parameter categories in FastAPI — path (required, in the URL), query (optional, after `?`), and body (JSON payload) — and combine them all in a single `PUT` route.


In [ ]:
%pip install -q fastapi uvicorn[standard] httpx


## Step 1 · Path Parameters — Typed, Required, In-URL

Path parameters are **embedded directly in the URL path** and are always required. They appear inside `{}` in the route template and must have a matching typed function parameter.

```
GET /users/42
            ^^ ← path parameter: user_id = 42
```

FastAPI supports all Python primitive types for path parameters:

| Type | URL segment | Result |
|---|---|---|
| `int` | `/items/7` | `7` (int) |
| `float` | `/prices/9.99` | `9.99` (float) |
| `str` | `/users/alice` | `"alice"` (str) |
| `bool` | `/flags/true` | `True` (bool) |
| `uuid.UUID` | `/objects/...` | UUID object |


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Parameter Demo API", version="1.0.0")

# In-memory user store for the examples throughout this notebook
USERS_DB = {
    1: {"id": 1, "name": "Alice", "role": "admin"},
    2: {"id": 2, "name": "Bob",   "role": "editor"},
    3: {"id": 3, "name": "Carol", "role": "viewer"},
}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    """Retrieve a single user by integer ID."""
    if user_id not in USERS_DB:
        # Return structured error — Day 3+ will use HTTPException for proper status codes
        return {"error": f"User {user_id} not found"}
    return USERS_DB[user_id]

client = TestClient(app)

# Valid integer ID
r = client.get("/users/1")
print("GET /users/1  →", r.status_code, r.json())

# Non-existent ID
r = client.get("/users/99")
print("GET /users/99 →", r.status_code, r.json())

# Non-integer ID — FastAPI returns 422 Unprocessable Entity automatically
r = client.get("/users/alice")
print("GET /users/alice → status:", r.status_code)
print("  validation error type:", r.json()["detail"][0]["type"])


### What just happened?

- **FastAPI validates the path parameter** before your function is ever called. If validation fails, the client receives a 422 with a detailed error — your route handler is not invoked.
- The **422 detail array** contains `loc` (which parameter failed), `msg` (human-readable reason), and `type` (machine-readable error code) — ideal for client-side error handling.
- **Type hint = single source of truth:** the same `user_id: int` declaration drives URL parsing, validation, OpenAPI schema generation, and editor autocomplete.
- Path parameters are **always required** — there is no way to make a path segment optional in FastAPI (use query params for optional values).


## Step 2 · Query Parameters — Optional, After the `?`

Query parameters appear **after the `?`** in the URL and are typically optional with defaults:

```
GET /items?skip=0&limit=10
           ^^^^   ^^^^^ ← query parameters
```

**Declaring query params** is identical to declaring any other function parameter — FastAPI infers they are query parameters because they are **not** in the path template:

```python
# path param:  user_id is in {user_id} → path parameter
# query param: skip and limit are NOT in the path → query parameters
@app.get("/users/{user_id}/items")
def get_user_items(user_id: int, skip: int = 0, limit: int = 10):
    ...
```


In [ ]:
from typing import Optional

# In-memory items store with role tags for filtering
ITEMS_DB = [
    {"id": i, "name": f"Item {i}", "category": ["tools", "parts", "extras"][i % 3]}
    for i in range(1, 21)  # 20 items
]

@app.get("/items")
def list_items(
    skip: int = 0,               # default=0: start from the beginning
    limit: int = 10,             # default=10: return up to 10 items
    category: Optional[str] = None,  # optional filter; None means "all categories"
):
    """List items with pagination and optional category filter."""
    # Apply optional category filter first
    filtered = ITEMS_DB
    if category is not None:
        filtered = [item for item in ITEMS_DB if item["category"] == category]
    # Then apply skip/limit pagination
    page = filtered[skip : skip + limit]
    return {"total": len(filtered), "skip": skip, "limit": limit, "items": page}

# Recreate the client to pick up the new route
client = TestClient(app)

# Default parameters: skip=0, limit=10, no filter
r = client.get("/items")
data = r.json()
print(f"GET /items            → total={data['total']}, returned={len(data['items'])}")

# Explicit pagination: second page of 5
r = client.get("/items?skip=5&limit=5")
data = r.json()
print(f"GET /items?skip=5&limit=5 → items: {[i['name'] for i in data['items']]}")

# Category filter
r = client.get("/items?category=tools")
data = r.json()
print(f"GET /items?category=tools → total={data['total']}, first: {data['items'][0]}")


### What just happened?

- **`Optional[str] = None`** (or equivalently `str | None = None` in Python 3.10+) makes a query param optional. Without a default, the param is required and FastAPI returns 422 if it's missing.
- **Pagination with `skip` and `limit`** is the standard REST pattern — `skip` is the offset, `limit` is the page size. FastAPI validates both as integers automatically.
- Query params are **type-checked too**: `?skip=abc` returns a 422 just like a bad path param.
- **`Optional` from `typing`** is equivalent to `X | None` in Python 3.10+. Both work in FastAPI.


## Step 3 · Request Body — JSON Payload via Pydantic

A **request body** is JSON data the client sends in the HTTP request body (typically with POST, PUT, PATCH). In FastAPI, you declare request bodies as **Pydantic models**:

```python
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True  # optional field with default
```

FastAPI detects that a function parameter is a Pydantic model and **automatically reads it from the request body** — no `request.json()` call needed.


In [ ]:
from pydantic import BaseModel

class ItemCreate(BaseModel):
    name: str                   # required — no default
    price: float                # required — no default
    category: str = "general"  # optional — has default
    in_stock: bool = True       # optional — has default

# Global counter to simulate auto-increment IDs
next_id = {"value": len(ITEMS_DB) + 1}

@app.post("/items", status_code=201)  # 201 Created is the correct status for POST
def create_item(item: ItemCreate):
    """Create a new item. Body must include name and price."""
    new_item = {"id": next_id["value"], **item.model_dump()}
    ITEMS_DB.append(new_item)
    next_id["value"] += 1
    return new_item

# Recreate client after adding the POST route
client = TestClient(app)

# POST with a full valid body
payload = {"name": "Widget", "price": 14.99, "category": "tools"}
r = client.post("/items", json=payload)
print("POST /items (full body)  →", r.status_code, r.json())

# POST with only required fields — defaults fill in the rest
r = client.post("/items", json={"name": "Gadget", "price": 5.0})
print("POST /items (min body)   →", r.status_code, r.json())

# POST with missing required field — 422 from FastAPI
r = client.post("/items", json={"name": "Broken"})
print("POST /items (missing price) → status:", r.status_code)
print("  error:", r.json()["detail"][0]["msg"])


### What just happened?

- **`item: ItemCreate`** in the function signature tells FastAPI: "read the request body, parse it as JSON, validate it against `ItemCreate`, and give me the resulting Pydantic model."
- **`model_dump()`** converts a Pydantic model to a plain Python dict — the Pydantic v2 name for what was `dict()` in v1.
- **`status_code=201`** in the decorator sets the HTTP response code for successful requests. FastAPI defaults to 200 — always set 201 for creation endpoints.
- Missing required fields and wrong types both produce **structured 422 errors** with the field path in the `loc` array, so clients know exactly which field failed.


## Step 4 · Combining Path + Query + Body in One Route

FastAPI determines parameter source **purely from context**:

| Declared as | Source |
|---|---|
| Matches a `{name}` in path template | Path parameter |
| Pydantic `BaseModel` subclass | Request body |
| Any other type with or without default | Query parameter |

This means you can have all three in a single function signature, and FastAPI handles each correctly:

```python
@app.put("/items/{item_id}")
def update_item(item_id: int, item: ItemUpdate, notify: bool = False):
    # item_id → path param
    # item    → request body (Pydantic model)
    # notify  → query param (?notify=true)
    ...
```


In [ ]:
class ItemUpdate(BaseModel):
    name: Optional[str] = None    # all fields optional for partial updates
    price: Optional[float] = None
    category: Optional[str] = None
    in_stock: Optional[bool] = None

@app.put("/items/{item_id}")
def update_item(
    item_id: int,              # ← path parameter (from URL template)
    item: ItemUpdate,          # ← request body  (Pydantic model)
    notify: bool = False,      # ← query parameter (?notify=true)
):
    """Update an existing item. Supports partial updates (PATCH semantics via PUT)."""
    # Find the item in ITEMS_DB list
    existing = next((i for i in ITEMS_DB if i["id"] == item_id), None)
    if existing is None:
        return {"error": f"Item {item_id} not found"}

    # Only update fields that were explicitly provided (not None)
    updates = item.model_dump(exclude_none=True)  # exclude_none drops None values
    existing.update(updates)

    # Simulate a notification side-effect
    response = {"updated": existing}
    if notify:
        response["notification"] = f"Admin notified about update to item {item_id}"

    return response

# Recreate client
client = TestClient(app)

# First, add an item to update
r = client.post("/items", json={"name": "Sensor", "price": 49.99})
new_id = r.json()["id"]
print(f"Created item ID={new_id}: {r.json()}")

# PUT with path + body only (no query param → notify defaults to False)
r = client.put(f"/items/{new_id}", json={"price": 39.99})
print(f"\nPUT /items/{new_id} (price update only) →", r.json())

# PUT with path + body + query param
r = client.put(f"/items/{new_id}?notify=true", json={"name": "Sensor Pro", "in_stock": False})
print(f"\nPUT /items/{new_id}?notify=true →", r.json())


### What just happened?

- **FastAPI automatically dispatches** each parameter to the correct source: `item_id` from the path, `item` from the JSON body, `notify` from the query string — all declared in one function signature.
- **`model_dump(exclude_none=True)`** is the standard Pydantic v2 pattern for partial updates: only fields the client actually sent are included; unset optional fields (which default to `None`) are excluded.
- **`?notify=true`** is automatically parsed as Python `True` (bool) because the parameter is typed `bool`. FastAPI accepts `true`, `True`, `1`, `on`, `yes` as truthy values for booleans.
- This single-function pattern is how FastAPI keeps route handlers **concise and readable** — no manual `request.args.get()` or `request.json()` calls.


## Step 5 · Request Body Echo — POST Round-Trip

A common pattern during API development is the **echo endpoint** — a route that returns the parsed and validated request body back to the caller. This confirms:
1. The body reached the server
2. FastAPI parsed and validated it correctly
3. Pydantic applied defaults for missing optional fields


In [ ]:
class EchoPayload(BaseModel):
    message: str
    count: int = 1
    tags: list[str] = []         # list field with default empty list
    metadata: dict = {}          # dict field with default empty dict

@app.post("/echo")
def echo(payload: EchoPayload):
    """Parse the request body and echo it back with a received timestamp."""
    import datetime
    return {
        "received": payload.model_dump(),   # the parsed + validated body
        "repeated": [payload.message] * payload.count,  # demonstrate count field
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }

client = TestClient(app)

# Minimal body — defaults fill in count, tags, metadata
r = client.post("/echo", json={"message": "hello"})
print("Minimal body response:")
import json
print(json.dumps(r.json(), indent=2))

# Full body with all fields
r = client.post("/echo", json={
    "message": "FastAPI",
    "count": 3,
    "tags": ["web", "python"],
    "metadata": {"source": "notebook", "day": 2},
})
print("\nFull body response:")
print(json.dumps(r.json(), indent=2))


### What just happened?

- **`list[str]` and `dict`** are valid Pydantic field types — FastAPI validates that the JSON array contains only strings, and that the metadata value is a JSON object.
- **Default empty collections** (`[]`, `{}`) must be set as defaults with care — Pydantic handles this correctly (no mutable default problem because Pydantic creates new instances per model).
- **`model_dump()`** serializes the model back to a Python dict that FastAPI then re-serializes to JSON — you can see Pydantic's applied defaults in the response.
- Echo endpoints are also useful for **testing client libraries** — send a known payload and verify the server parses it exactly as expected.


In [ ]:
# Challenge: Build a PATCH /users/{user_id} route that:
#   1. Takes user_id as a path parameter (int)
#   2. Accepts an optional query parameter: send_welcome (bool, default False)
#   3. Accepts a request body: UserPatch model with optional fields: name (str), role (str)
#   4. If the user doesn't exist, return {"error": "User N not found"}
#   5. Only update fields that were provided (use exclude_none=True)
#   6. If send_welcome is True, add "welcome_email_sent": True to the response
#   7. Test: patch user 1's name to "Alexandra", with send_welcome=true

# Your solution here:

# class UserPatch(BaseModel):
#     ...

# @app.patch("/users/{user_id}")
# def patch_user(user_id: int, user: UserPatch, send_welcome: bool = False):
#     ...

# client = TestClient(app)
# r = client.patch("/users/1?send_welcome=true", json={"name": "Alexandra"})
# print(r.json())


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Path parameter | In `{name}` in route template; always required; validated by type hint |
| Query parameter | Not in path template; optional when it has a default; after `?` in URL |
| Request body | Pydantic `BaseModel` subclass in function signature; read from JSON body |
| `Optional[T] = None` | Makes query param (or body field) optional — equivalent to `T \| None = None` |
| `model_dump()` | Converts Pydantic model to dict; use `exclude_none=True` for partial updates |
| Status codes | Set `status_code=201` for creation; FastAPI defaults to 200 |
| Combining all three | Path + query + body can coexist in one function signature; FastAPI routes each automatically |

> **Tip:** FastAPI uses Python type hints as the single source of truth for parameter types, validation, serialization, and documentation.

---
## What's next
**Day 3** → Deep-dive into Pydantic models: field-level validation constraints, nested models, `@field_validator`, and `model_config`.

Mark Day 2 complete in your [tracker](../index.html).
